# Chicago Traffic Crashes: cleaning notebook

**One "Run all" rebuilds everything** from the raw CSV (`data/export.csv`):

* `output/chicago_crashes_clean.csv`: cleaned data (42 columns)
* `output/drop_summary.csv`: every dropped row with the rule ID it failed
* `output/out_of_bounds_points.csv`: points outside the Chicago bounding box
* `output/export_manifest.json` + `output/cleaning_report.md`: counts per rule ID
* `docs/data/*`: the compact data the website loads

All the logic lives in `pipeline/crash_pipeline.py`, so the notebook, the tests and the website export share one implementation. Every rule has an ID (**R01–R16**) that appears here, in the export files and in the website's *Methods* panel.

Requirements: Python 3.9+, `pandas`, `numpy`, `matplotlib` (only for its fast point-in-polygon routine).

In [ ]:
import os, sys, json, time
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebook" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "pipeline"))
import crash_pipeline as cp

RAW   = os.path.join(ROOT, "data", "export.csv")
OUT   = os.path.join(ROOT, "output")
SITE  = os.path.join(ROOT, "docs", "data")
NEIGH = os.path.join(ROOT, "data", "igwz-8jzy.geojson")
ZIPS  = os.path.join(ROOT, "data", "unjd-c2ca.geojson")
pd.set_option("display.max_colwidth", 120)
print("project root:", ROOT)

## Rules and their IDs

In [ ]:
pd.DataFrame([{"ID": k, "Name": v["name"], "Kind": v["kind"], "Spec section": v["spec"], "Rule": v["text"]}
              for k, v in cp.RULES.items()])

## Raw data at a glance
The raw file is read in chunks (it is ~700 MB), so this needs ~1.5 GB of RAM.

In [ ]:
head = pd.read_csv(RAW, nrows=5, dtype=str)
print(len(head.columns), "columns")
head.T.head(49)

## Run the full pipeline (R01–R16)

Drop rules are checked in the order **R01 → R02 → R04 → R05 → R07 → R14**. The first one a row fails is its `RULE_ID`, and `ALL_FAILED_RULES` lists every one it fails. The input file is only read, never written.

In [ ]:
t0 = time.time()
manifest = cp.run(RAW, OUT, site_dir=SITE, neighborhoods_path=NEIGH, zips_path=ZIPS, verbose=True)
print(f"done in {time.time()-t0:.0f} s")
print(f"rows in {manifest['rows_in']:,} -> kept {manifest['rows_out']:,}, dropped {manifest['rows_dropped']:,}")

In [ ]:
pd.DataFrame(manifest["rules"])[["id", "name", "kind", "rows_dropped", "rows_failing_incl_secondary"]]

In [ ]:
drops = pd.read_csv(os.path.join(OUT, cp.DROP_FILE), dtype=str)
clean = pd.read_csv(os.path.join(OUT, cp.CLEAN_FILE), dtype=str, keep_default_na=False)
print(clean.shape)

### R01: Date range

In [ ]:
drops[drops.RULE_ID == "R01"].head(3)

In [ ]:
print(clean.CRASH_DATE.min(), "->", clean.CRASH_DATE.max())

### R02: Duplicate CRASH_RECORD_ID

In [ ]:
print("duplicates dropped:", (drops.RULE_ID == "R02").sum())

In [ ]:
assert not clean.CRASH_RECORD_ID.duplicated().any()

### R03: Dropped columns

In [ ]:
print(cp.DROPPED_COLUMNS)

In [ ]:
assert not set(cp.DROPPED_COLUMNS) & set(clean.columns); print(len(clean.columns), "columns")

### R04: Coordinates (missing / zero / outside box)

In [ ]:
drops[drops.RULE_ID == "R04"].DETAIL.str.replace(r"\(.*\)", "", regex=True).value_counts()

In [ ]:
oob = pd.read_csv(os.path.join(OUT, cp.OOB_FILE), dtype=str)
print(len(oob), "points outside the bounding box (non-missing, non-zero)")
oob.head(20)

### R05: Late police notification (> 30 days)

In [ ]:
drops[drops.RULE_ID == "R05"].head(5)

### R06: Y/N indicators → YES/NO

In [ ]:
clean[[c for c in clean.columns if c.endswith("_I")]].apply(pd.Series.value_counts)

### R07: Posted speed limit

In [ ]:
drops[drops.RULE_ID == "R07"].DETAIL.value_counts().head(15)

In [ ]:
clean.POSTED_SPEED_LIMIT.value_counts()

### R08: Missing injuries → 0

In [ ]:
print("blank injury cells set to 0:", manifest["value_changes"]["R08_blank_to_0"])

### R09: NUM_UNITS buckets

In [ ]:
clean.NUM_UNITS.value_counts()

### R10: Missing values (work zone, flags, beat)

In [ ]:
{k: v for k, v in manifest["value_changes"].items() if k.startswith("R10")}

In [ ]:
clean.WORK_ZONE_TYPE.value_counts()

### R11: KSI

In [ ]:
(clean.KSI.astype(int) == clean.INJURIES_FATAL.astype(int) + clean.INJURIES_INCAPACITATING.astype(int)).all()

In [ ]:
clean.KSI.value_counts()

### R12: NEIGHBORHOOD (community areas)

In [ ]:
clean.NEIGHBORHOOD.value_counts().head(10)

In [ ]:
print(clean.NEIGHBORHOOD.nunique(), "areas used")

### R13: ZIPCODE

In [ ]:
clean.ZIPCODE.value_counts().head(10)

In [ ]:
print(clean.ZIPCODE.nunique(), "ZIP codes used")

### R14: Outside city boundaries (added rule)

In [ ]:
drops[drops.RULE_ID == "R14"].head(5)

### R15: Other missing values (added rule)

In [ ]:
{k: v for k, v in manifest["value_changes"].items() if k.startswith("R15")}

### R16: Formats and deterministic order (added rule)

In [ ]:
clean[["CRASH_DATE", "CRASH_RECORD_ID"]].head()

In [ ]:
print(json.dumps(manifest["outputs"], indent=1))

## Verify
These run the same tests as `python3 -m unittest discover -s tests`.

In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover(os.path.join(ROOT, "tests"), pattern="test_*.py")
unittest.TextTestRunner(verbosity=1).run(suite)

## Site data
`docs/data/` is what the browser downloads. NFR-2 requires it to be ≤ 10 MB.

In [ ]:
sizes = {f: os.path.getsize(os.path.join(SITE, f)) for f in sorted(os.listdir(SITE))}
print({k: f"{v/1e6:.2f} MB" for k, v in sizes.items()}, "total", f"{sum(sizes.values())/1e6:.2f} MB")